# M3 · Optimization

**Outcome:** Compare search directions, learning rates, and momentum before fitting a model.

Run cells with **Shift + Enter**. PyTorch is already installed in standard Colab runtimes.

## Experiment question

> Which information makes an update efficient and stable?

Before running code, write a prediction. Then observe the evidence, change one variable, and explain the difference.

In [ ]:
import torch
print('PyTorch', torch.__version__)
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')

## 1 · Random directions versus the gradient
On a quadratic, compare equal-length moves. As dimension grows, a random unit vector is usually poorly aligned with the negative gradient.

In [ ]:
torch.manual_seed(7)
for dimension in [2, 10, 100, 1000]:
    w = torch.ones(dimension) / dimension**0.5
    directions = torch.randn(120, dimension)
    directions = directions / directions.norm(dim=1, keepdim=True)
    random_change = 0.1 * (directions @ w) + 0.5 * 0.1**2
    gradient_change = 0.5 * (1 - 0.1)**2 - 0.5
    print(dimension, 'random best:', random_change.min().item(), 'gradient:', gradient_change)

## 2 · Learning rate and momentum
Inspect the gradient and the stored velocity separately. Predict the next parameter before running the loop.

In [ ]:
p = torch.tensor(1.0)
velocity = torch.tensor(0.0)
lr, beta = 0.1, 0.9
for step in range(3):
    gradient = p.clone()       # gradient of 1/2 p²
    velocity = beta * velocity + gradient
    p = p - lr * velocity
    print(step + 1, 'gradient:', gradient.item(), 'velocity:', velocity.item(), 'p:', p.item())

## 3 · Put the optimizer inside a training loop

In [ ]:
x = torch.linspace(-2, 2, 80).unsqueeze(1)
y = 1.5*x - .3 + .15*torch.randn_like(x)
model = torch.nn.Linear(1, 1)
loss_fn = torch.nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=.08)

In [ ]:
for epoch in range(80):
    optimizer.zero_grad()
    loss = loss_fn(model(x), y)
    loss.backward()
    optimizer.step()
print('loss:', loss.item())
print('weight:', model.weight.item(), 'bias:', model.bias.item())

## Reflection

1. What did you predict?
2. What evidence did the output provide?
3. Which one variable did you change?
4. How does the result connect to the lesson's mental model?